# 가상 설비별 데이터 전처리·Custom Accident 생성·ML 학습

이 노트북은 원본 데이터를 4개 가상 설비에 맞게 보정하고 합성 정답 `custom_accident`를 생성합니다. 모델 비교와 평가 후 데모에서 사용할 가공 CSV, 학습된 모델, 임계값 메타데이터를 저장합니다.


In [ ]:
from pathlib import Path
import json
import joblib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

RANDOM_STATE = 42
DATA_PATH = Path('data/industrial_fire_risk_data.csv')
rng = np.random.default_rng(RANDOM_STATE)

## 1. 원본 데이터 확인

In [ ]:
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)

quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_rate': df.isna().mean(),
    'unique_count': df.nunique(dropna=False),
})
print(f'데이터 크기: {len(df):,}행 × {df.shape[1]}열')
print(f"원본 Accident 비율: {df['Accident'].mean():.2%}")
display(df.head())
display(quality)

## 2. 가상 설비 배정

각 Factory 안에서 관측치를 3개 설비에 재현 가능하게 배정합니다. 설비 유형마다 위험 규칙이 다르며 `rule_version`과 `manual_id`는 이후 RAG Agent가 설비별 매뉴얼을 찾는 키로 사용할 수 있습니다.

In [ ]:
machine_type_map = {
    'M-0101': 'REACTOR',
    'M-0102': 'COMPRESSOR',
    'M-0103': 'STORAGE_TANK',
    'M-0104': 'PUMP',
}
machine_ids = list(machine_type_map)
assignment = np.resize(np.array(machine_ids), len(df))
rng.shuffle(assignment)
df['machine_id'] = assignment
df['machine_type'] = df['machine_id'].map(machine_type_map)

# 사고 라벨 생성 전에 기계별 센서 특성을 보정합니다.
load_factor = rng.normal(0, 1, len(df))
reactor = df['machine_id'].eq('M-0101')
compressor = df['machine_id'].eq('M-0102')
tank = df['machine_id'].eq('M-0103')
pump = df['machine_id'].eq('M-0104')
df.loc[reactor, 'Temp'] = df.loc[reactor, 'Temp']*1.10 + 6 + 2.0*load_factor[reactor]
df.loc[reactor, 'Pressure'] = df.loc[reactor, 'Pressure']*1.12 + 1.5*load_factor[reactor]
df.loc[reactor, 'Vibration'] *= 0.75
df.loc[reactor, 'Speed'] *= 0.65
df.loc[reactor, 'Gas'] *= 1.05
df.loc[compressor, 'Temp'] = df.loc[compressor, 'Temp'] + 2 + 0.8*load_factor[compressor]
df.loc[compressor, 'Pressure'] = df.loc[compressor, 'Pressure']*1.10 + 1.2*load_factor[compressor]
df.loc[compressor, 'Vibration'] = df.loc[compressor, 'Vibration']*1.20 + 0.35*load_factor[compressor]
df.loc[compressor, 'Speed'] = df.loc[compressor, 'Speed']*1.15 + 220*load_factor[compressor]
df.loc[compressor, 'Gas'] *= 0.70
df.loc[tank, 'Temp'] = df.loc[tank, 'Temp']*0.88 + 1.2*load_factor[tank]
df.loc[tank, 'Pressure'] *= 0.90
df.loc[tank, 'Vibration'] *= 0.50
df.loc[tank, 'Speed'] *= 0.40
df.loc[tank, 'Gas'] = df.loc[tank, 'Gas']*1.15 + 0.45*load_factor[tank]
df.loc[pump, 'Temp'] = df.loc[pump, 'Temp'] + 0.8*load_factor[pump]
df.loc[pump, 'Pressure'] = df.loc[pump, 'Pressure']*0.88 + 0.8*load_factor[pump]
df.loc[pump, 'Vibration'] = df.loc[pump, 'Vibration']*1.10 + 0.25*load_factor[pump]
df.loc[pump, 'Speed'] = df.loc[pump, 'Speed']*0.90 + 160*load_factor[pump]
df.loc[pump, 'Gas'] *= 0.60
df['Temp'] = df['Temp'].clip(5, 60)
df['Pressure'] = df['Pressure'].clip(5, 60)
df['Vibration'] = df['Vibration'].clip(0, 8)
df['Speed'] = df['Speed'].clip(300, 5000)
df['Gas'] = df['Gas'].clip(0, 10)

profile_rows = []
for i, machine_id in enumerate(machine_ids):
    machine_type = machine_type_map[machine_id]
    profile_rows.append({
        'machine_id': machine_id,
        'machine_type': machine_type,
        'rule_version': f'{machine_type[:3]}-R1',
        'manual_id': f'{machine_type.lower()}_safety_manual',
        'risk_bias': rng.normal(0, 0.18),
    })
machine_profiles = pd.DataFrame(profile_rows)
df = df.merge(machine_profiles, on=['machine_id', 'machine_type'], how='left')
display(machine_profiles)
display(pd.crosstab(df['Factory'], df['machine_type']))

## 3. 설비별 확률적 `custom_accident` 생성

각 수치형 변수는 전체 데이터의 중앙값과 90% 분위수를 이용해 위험도를 0~2 범위로 변환합니다. 설비별로 단일 변수뿐 아니라 상호작용 항을 다르게 적용하고, 관측되지 않은 현장 요인을 표현하는 잡음을 추가합니다.

In [ ]:
def high_risk(series):
    q50, q90 = series.quantile([0.50, 0.90])
    scale = max(q90 - q50, 1e-9)
    return ((series - q50) / scale).clip(0, 2)

def low_risk(series):
    q10, q50 = series.quantile([0.10, 0.50])
    scale = max(q50 - q10, 1e-9)
    return ((q50 - series) / scale).clip(0, 2)

risk = pd.DataFrame(index=df.index)
risk['temp'] = high_risk(df['Temp'])
risk['pressure_high'] = high_risk(df['Pressure'])
risk['pressure_low'] = low_risk(df['Pressure'])
risk['humidity_low'] = low_risk(df['Humidity'])
risk['vibration'] = high_risk(df['Vibration'])
risk['speed'] = high_risk(df['Speed'])
risk['age'] = high_risk(df['Age'])
risk['service'] = high_risk(df['Service_Days'])
risk['gas'] = high_risk(df['Gas'])
risk['sparks'] = (df['Sparks'] / max(df['Sparks'].max(), 1)).clip(0, 1)
risk['crowding'] = high_risk(df['Workers'].fillna(df['Workers'].median()))
risk['night'] = df['Shift'].eq('Night').astype(float)
risk['untrained'] = df['Training'].eq('No').astype(float)
risk['junior'] = df['Exp'].eq('Junior').astype(float)

score = pd.Series(-4.20, index=df.index, dtype=float) + df['risk_bias']

reactor = df['machine_type'].eq('REACTOR')
score.loc[reactor] += (
    0.95*risk.loc[reactor, 'temp']
    + 0.75*risk.loc[reactor, 'pressure_high']
    + 0.70*(risk.loc[reactor, 'temp'] * risk.loc[reactor, 'pressure_high'])
    + 0.25*risk.loc[reactor, 'service']
    + 0.25*risk.loc[reactor, 'night']
)

compressor = df['machine_type'].eq('COMPRESSOR')
score.loc[compressor] += (
    0.90*risk.loc[compressor, 'vibration']
    + 0.65*risk.loc[compressor, 'pressure_high']
    + 0.55*risk.loc[compressor, 'speed']
    + 0.65*(risk.loc[compressor, 'vibration'] * risk.loc[compressor, 'speed'])
    + 0.25*risk.loc[compressor, 'age']
)

tank = df['machine_type'].eq('STORAGE_TANK')
score.loc[tank] += (
    0.90*risk.loc[tank, 'gas']
    + 0.65*risk.loc[tank, 'temp']
    + 0.75*risk.loc[tank, 'sparks']
    + 0.80*(risk.loc[tank, 'gas'] * risk.loc[tank, 'sparks'])
    + 0.25*risk.loc[tank, 'humidity_low']
)

pump = df['machine_type'].eq('PUMP')
score.loc[pump] += (
    0.85*risk.loc[pump, 'vibration']
    + 0.65*risk.loc[pump, 'service']
    + 0.55*risk.loc[pump, 'age']
    + 0.60*risk.loc[pump, 'pressure_low']
    + 0.55*(risk.loc[pump, 'vibration'] * risk.loc[pump, 'service'])
)

# 모든 설비에 공통으로 적용되는 작업환경 위험
score += (
    0.20*risk['untrained']
    + 0.15*risk['junior']
    + 0.15*risk['crowding']
    + rng.normal(0, 0.35, size=len(df))
)

df['risk_score'] = score
df['accident_probability'] = 1 / (1 + np.exp(-df['risk_score']))
df['custom_accident'] = rng.binomial(1, df['accident_probability'])

label_summary = df.groupby('machine_type').agg(
    rows=('custom_accident', 'size'),
    custom_accident_rate=('custom_accident', 'mean'),
    mean_probability=('accident_probability', 'mean'),
    original_accident_rate=('Accident', 'mean'),
).sort_values('custom_accident_rate', ascending=False)
print(f"전체 custom_accident 비율: {df['custom_accident'].mean():.2%}")
display(label_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
label_summary['custom_accident_rate'].sort_values().plot.barh(ax=axes[0], color='#C44E52')
axes[0].set_title('Custom accident rate by machine type')
axes[0].set_xlabel('Rate')
df['accident_probability'].hist(ax=axes[1], bins=40, color='#4C72B0', edgecolor='white')
axes[1].set_title('Generated accident probability')
axes[1].set_xlabel('Probability')
plt.tight_layout()
plt.show()

## 4. 학습 데이터와 시간 순서 분할

원본 결과 컬럼과 라벨 생성 내부값은 입력에서 제외합니다. 과거 70%로 학습하고 다음 15%로 임계값을 선택하며 가장 최근 15%를 최종 테스트합니다.

In [ ]:
numeric_features = [
    'Workers', 'Temp', 'Pressure', 'Humidity', 'Vibration',
    'Speed', 'Age', 'Service_Days', 'Gas', 'Sparks',
]
categorical_features = [
    'Factory', 'Region', 'Shift', 'Exp', 'Training',
    'machine_id', 'machine_type',
]
feature_columns = numeric_features + categorical_features
leakage_columns = [
    'Accident', 'Risk', 'Alarm', 'risk_score',
    'accident_probability', 'custom_accident',
]
assert not set(feature_columns) & set(leakage_columns)

n = len(df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)
train_df = df.iloc[:train_end]
valid_df = df.iloc[train_end:valid_end]
test_df = df.iloc[valid_end:]

X_train, y_train = train_df[feature_columns], train_df['custom_accident']
X_valid, y_valid = valid_df[feature_columns], valid_df['custom_accident']
X_test, y_test = test_df[feature_columns], test_df['custom_accident']

split_summary = pd.DataFrame({
    'rows': [len(train_df), len(valid_df), len(test_df)],
    'positive_rate': [y_train.mean(), y_valid.mean(), y_test.mean()],
    'start': [train_df.timestamp.min(), valid_df.timestamp.min(), test_df.timestamp.min()],
    'end': [train_df.timestamp.max(), valid_df.timestamp.max(), test_df.timestamp.max()],
}, index=['train', 'validation', 'test'])
display(split_summary)

## 5. Logistic Regression vs Random Forest

Logistic Regression은 해석 가능한 선형 베이스라인이고, Random Forest는 설비별 비선형 조건과 변수 간 상호작용을 포착하는 비교 모델입니다.

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

models = {
    'Logistic Regression': Pipeline([
        ('preprocess', preprocessor),
        ('model', LogisticRegression(
            max_iter=2_000, class_weight='balanced', random_state=RANDOM_STATE
        )),
    ]),
    'Random Forest': Pipeline([
        ('preprocess', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=100, min_samples_leaf=5, max_features='sqrt',
            class_weight='balanced_subsample', random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
}

validation_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_valid)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    validation_rows.append({
        'model': name,
        'roc_auc': roc_auc_score(y_valid, probability),
        'pr_auc': average_precision_score(y_valid, probability),
        'recall_at_0.5': recall_score(y_valid, prediction),
        'precision_at_0.5': precision_score(y_valid, prediction, zero_division=0),
        'balanced_accuracy_at_0.5': balanced_accuracy_score(y_valid, prediction),
    })
validation_results = pd.DataFrame(validation_rows).sort_values('pr_auc', ascending=False)
display(validation_results)

## 6. 검증 데이터로 경고 임계값 선택

산업안전에서는 사고 누락 비용이 크므로 F1보다 Recall에 가중치를 둔 F2 점수가 가장 높은 임계값을 선택합니다. 최종 테스트 데이터는 임계값 선택에 사용하지 않습니다.

In [ ]:
best_model_name = validation_results.iloc[0]['model']
best_model = models[best_model_name]
valid_probability = best_model.predict_proba(X_valid)[:, 1]

threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.01):
    prediction = (valid_probability >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_valid, prediction, zero_division=0),
        'recall': recall_score(y_valid, prediction, zero_division=0),
        'f2': fbeta_score(y_valid, prediction, beta=2, zero_division=0),
    })
threshold_results = pd.DataFrame(threshold_rows)
best_threshold = threshold_results.loc[threshold_results['f2'].idxmax(), 'threshold']
print(f'선택 모델: {best_model_name}')
print(f'선택 임계값: {best_threshold:.2f}')
display(threshold_results.sort_values('f2', ascending=False).head(10))

test_probability = best_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= best_threshold).astype(int)
test_metrics = pd.Series({
    'roc_auc': roc_auc_score(y_test, test_probability),
    'pr_auc': average_precision_score(y_test, test_probability),
    'precision': precision_score(y_test, test_prediction, zero_division=0),
    'recall': recall_score(y_test, test_prediction, zero_division=0),
    'f2': fbeta_score(y_test, test_prediction, beta=2, zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_prediction),
}, name='test')
display(test_metrics.to_frame())
print(classification_report(y_test, test_prediction, digits=4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, test_prediction)).plot(ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[2])
axes[0].set_title('Confusion matrix')
axes[1].set_title('ROC curve')
axes[2].set_title('Precision-Recall curve')
plt.tight_layout()
plt.show()

## 7. 원본 컬럼 기준 Permutation Importance

테스트 표본에서 컬럼 하나씩 섞었을 때 PR-AUC가 얼마나 감소하는지 계산합니다. 양의 감소폭이 클수록 모델이 해당 컬럼에 더 의존합니다. 이는 인과관계가 아닙니다.

In [ ]:
importance_sample = X_test.sample(min(5_000, len(X_test)), random_state=RANDOM_STATE)
importance_y = y_test.loc[importance_sample.index]
importance = permutation_importance(
    best_model, importance_sample, importance_y, scoring='average_precision',
    n_repeats=3, random_state=RANDOM_STATE, n_jobs=1,
)
importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance_mean': importance.importances_mean,
    'importance_std': importance.importances_std,
}).sort_values('importance_mean', ascending=False)
display(importance_df)

plot_df = importance_df.head(15).sort_values('importance_mean')
ax = plot_df.plot.barh(
    x='feature', y='importance_mean', xerr='importance_std',
    figsize=(10, 6), legend=False, color='#4C72B0', capsize=3,
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Permutation importance ({best_model_name})')
ax.set_xlabel('PR-AUC decrease after permutation')
plt.tight_layout()
plt.show()

## 8. Agent에서 사용할 사고확률 예측 함수

예측 결과는 확률과 경고 여부만 반환합니다. 실제 서비스에서는 `machine_id`로 `manual_id`를 조회해 설비별 RAG 검색과 연결할 수 있습니다.

In [ ]:
def predict_accident(input_rows, model=best_model, threshold=best_threshold):
    input_frame = pd.DataFrame(input_rows).copy()
    missing = set(feature_columns) - set(input_frame.columns)
    if missing:
        raise ValueError(f'필수 입력 컬럼이 없습니다: {sorted(missing)}')
    probability = model.predict_proba(input_frame[feature_columns])[:, 1]
    output = input_frame[['machine_id', 'machine_type']].copy()
    output['accident_probability'] = probability
    output['risk_percent'] = (probability * 100).round(1)
    output['accident_alert'] = (probability >= threshold).astype(int)
    output['threshold'] = threshold
    return output

demo_input = X_test.head(5).to_dict('records')
display(predict_accident(demo_input))

print('주의: custom_accident는 설비별 규칙으로 생성한 합성 라벨입니다.')
print('risk_score, accident_probability, 원본 Accident/Risk/Alarm은 모델 입력에 포함되지 않았습니다.')

## 9. 데모용 가공 데이터·모델·메타데이터 저장

데모 노트북은 원본 CSV를 다시 가공하거나 모델을 다시 학습하지 않습니다. 이 셀에서 생성한 세 가지 산출물만 불러와 새 Mock IoT 입력을 예측합니다.


In [ ]:
PROCESSED_PATH = Path('data/processed_industrial_fire_ml.csv')
MODEL_DIR = Path('models')
MODEL_PATH = MODEL_DIR / 'industrial_fire_accident_pipeline.joblib'
METADATA_PATH = MODEL_DIR / 'industrial_fire_accident_metadata.json'

manual_id_map = {
    'REACTOR': 'reactor_safety_manual',
    'COMPRESSOR': 'compressor_safety_manual',
    'STORAGE_TANK': 'storage_tank_safety_manual',
    'PUMP': 'pump_safety_manual',
}

processed_df = df[['timestamp'] + feature_columns + ['custom_accident']].copy()
processed_df.insert(3, 'manual_id', processed_df['machine_type'].map(manual_id_map))
processed_df = processed_df.sort_values(['timestamp', 'machine_id']).reset_index(drop=True)

forbidden_columns = {
    'Accident', 'Risk', 'Alarm', 'risk_score',
    'accident_probability', 'risk_bias', 'rule_version',
}
assert forbidden_columns.isdisjoint(processed_df.columns)
assert processed_df['custom_accident'].isin([0, 1]).all()

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(PROCESSED_PATH, index=False, encoding='utf-8-sig')
joblib.dump(best_model, MODEL_PATH)

metadata = {
    'model_name': best_model_name,
    'model_file': MODEL_PATH.name,
    'processed_data_file': PROCESSED_PATH.name,
    'target_column': 'custom_accident',
    'feature_columns': feature_columns,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'warning_threshold': float(best_threshold),
    'caution_threshold': float(best_threshold * 0.5),
    'random_state': RANDOM_STATE,
    'train_rows': int(len(train_df)),
    'validation_rows': int(len(valid_df)),
    'test_rows': int(len(test_df)),
    'test_metrics': {key: float(value) for key, value in test_metrics.items()},
    'manual_ids': manual_id_map,
    'notes': '합성 custom_accident 분류 모델이며 출력값은 실제 사고확률이 아닌 ML 위험 점수입니다.',
}
METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)

print(f'가공 CSV 저장: {PROCESSED_PATH.resolve()}')
print(f'학습 모델 저장: {MODEL_PATH.resolve()}')
print(f'메타데이터 저장: {METADATA_PATH.resolve()}')
print(f'가공 데이터 크기: {processed_df.shape[0]:,}행 × {processed_df.shape[1]}열')
display(pd.Series(metadata).to_frame('value'))
